In [ ]:
import coiled
import xarray as xr
import zarr

from srm import catalog

zarr.config.set({"async.concurrency": 64})

In [ ]:
cluster = coiled.Cluster(
    name="srm-fine-to-coarse",
    n_workers=[20, 40],
    region="us-west-2",
    worker_vm_types="m8g.large",
    scheduler_vm_types=["c8g.xlarge"],
    spot_policy="spot_with_fallback",
    tags={"Project": "SRM"},
)

client = cluster.get_client()
client

[2026-02-05 12:59:01,200][INFO    ][coiled] Fetching latest package priorities...
[2026-02-05 12:59:01,202][INFO    ][coiled.package_sync] Resolving your local /Users/nrhagen/Documents/carbonplan/SRM/uv.lock Python environment...
[2026-02-05 12:59:01,390][INFO    ][coiled.package_sync] Scanning 236 python packages...
[2026-02-05 12:59:01,659][INFO    ][coiled] Running pip check...
[2026-02-05 12:59:01,947][INFO    ][coiled] Validating environment...
[2026-02-05 12:59:02,624][INFO    ][coiled] Creating wheel for ~/Documents/carbonplan/SRM/src...
[2026-02-05 12:59:02,700][INFO    ][coiled] Creating wheel for srm...
[2026-02-05 12:59:04,484][INFO    ][coiled] Creating wheel for xarray-regrid...
[2026-02-05 12:59:10,916][INFO    ][coiled] Uploading coiled_local_src...
[2026-02-05 12:59:11,882][INFO    ][coiled] Uploading srm...
[2026-02-05 12:59:12,828][INFO    ][coiled] Uploading xarray-regrid...
[2026-02-05 12:59:13,768][INFO    ][coiled] Creating software environment...
[2026-02-05 12:5

<Client: 'tls://10.1.38.145:8786' processes=18 threads=36, memory=128.46 GiB>

## This shouls all be lazy and only take a few seconds

In [ ]:
# coarse historical GCM grid - Note, single time slice, single var
ds_coarse_grid = catalog.get("CESM2-WACCM-Historical-icechunk").to_xarray()[["tasmax"]].isel(time=0)

# fine ERA5 - single var
ds_fine_grid = catalog.get("ERA5").to_xarray()[["tasmax"]]

# create a target grid from the GCM coarse dataset
target_grid = ds_coarse_grid[["lat", "lon"]].drop_vars("time").reset_coords(drop=True)

# use xarray regird
ds_fine_regridded = ds_fine_grid.regrid.conservative(target_grid, latitude_coord="lat")

## Calling to_zarr or to_icechunk would trigger the computation. 

In [ ]:
%%time
# ~5 minutes
# We could speed this up with obstore + zarr backend
ds_fine_regridded.to_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", mode="w")

/Users/nrhagen/Documents/carbonplan/SRM/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/nrhagen/Documents/carbonplan/SRM/.venv/lib/python3.13/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 14.40 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: user 18.4 s, sys: 563 ms, total: 18.9 s
Wall time: 4min 45s


In [ ]:
# shutdown our coiled cluster
client.shutdown()

2026-02-05 13:05:52,050 - distributed.deploy.adaptive - INFO - Adaptive scaling stopped: minimum=20 maximum=40. Reason: unknown
[2026-02-05 13:05:52,240][INFO    ][coiled] Cluster 1421479 deleted successfully.


In [7]:
rtds = xr.open_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", chunks="auto")

In [ ]:
rtds.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()

In [ ]:
ds_coarse_grid["tasmax"].sel(lat=slice(25, 50), lon=slice(-100, -70)).plot()

In [ ]:
ds_fine_grid.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()